# Compreensão e preparação dos dados
Nesta etapa, as fontes do Censo Escolar, Taxas de Rendimento e Média de Alunos por Turma são inventariadas, compreendidas e validadas. Também são definidos os mapeamentos históricos, o recorte das escolas públicas de Ensino Fundamental de Porto Alegre e as regras de qualidade necessárias para preparar os dados de 2018 a 2023 para a limpeza e integração posteriores.


## Configuração do ambiente e dos caminhos
Este código importa as bibliotecas utilizadas no notebook, define os parâmetros gerais da análise (período, município, estado, redes públicas e tamanho dos blocos de leitura) e localiza automaticamente a raiz do projeto.
A partir da raiz, são criados caminhos padronizados para as pastas de dados brutos, documentação e dados processados. Isso torna o notebook mais portátil, evita caminhos repetidos e permite sua execução em diferentes ambientes sem alterar o código.

In [1]:
from pathlib import Path
import os
import re
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

ANOS = tuple(range(2018, 2024))
CHUNK_SIZE = 50_000
MUNICIPIO_ALVO = "Porto Alegre"
UF_ALVO = "RS"
DEPENDENCIAS_PUBLICAS = {"1", "2", "3"}  # federal, estadual, municipal

def encontrar_raiz_projeto(inicio=None):
    override = os.environ.get("PROJECT_ROOT_OVERRIDE")
    if override:
        raiz = Path(override).expanduser().resolve()
        if not (raiz / "data" / "raw").exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE inválido: {raiz}")
        return raiz
    inicio = Path(inicio or Path.cwd()).resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / "data" / "raw").exists():
            return candidato
    raise FileNotFoundError(
        "Raiz não encontrada. Execute o notebook dentro do repositório ou defina "
        "a variável de ambiente PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
RAW = PROJECT_ROOT / "data" / "raw"
DOCS = PROJECT_ROOT / "docs"
PROCESSED = PROJECT_ROOT / "data" / "processed"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Dados brutos: {RAW}")

Raiz do projeto: /media/eduardablanco/Novo volume/Projeto-Em-Business-Intelligence-e-Analytics
Dados brutos: /media/eduardablanco/Novo volume/Projeto-Em-Business-Intelligence-e-Analytics/data/raw


## Inventário dos arquivos brutos
Este código percorre todos os arquivos da pasta data/raw, identifica o ano presente em cada caminho e registra informações como fonte, nome, extensão, tamanho e localização.
Em seguida, apresenta o inventário completo e um resumo da quantidade e do tamanho dos arquivos por fonte e ano. Essa verificação ajuda a confirmar a disponibilidade dos dados de 2018 a 2023 antes do carregamento e processamento.

In [2]:
def extrair_ano(caminho):
    match = re.search(r"20\d{2}", str(caminho))
    return int(match.group()) if match else pd.NA

def inventariar_arquivos(raw=RAW):
    registros = []
    for arquivo in sorted(p for p in raw.rglob("*") if p.is_file()):
        registros.append({
            "base": arquivo.relative_to(raw).parts[0],
            "ano": extrair_ano(arquivo),
            "arquivo": arquivo.name,
            "extensao": arquivo.suffix.lower(),
            "tamanho_mb": round(arquivo.stat().st_size / 1024**2, 2),
            "caminho_relativo": str(arquivo.relative_to(PROJECT_ROOT)),
        })
    return pd.DataFrame(registros).sort_values(["base", "ano", "arquivo"], na_position="last")

inventario = inventariar_arquivos()
display(inventario)
display(inventario.groupby(["base", "ano"], dropna=False).agg(
    arquivos=("arquivo", "count"), tamanho_mb=("tamanho_mb", "sum")
).reset_index())

,base,ano,arquivo,extensao,tamanho_mb,caminho_relativo
1,censo_escolar,2018,Aluno.pdf,.pdf,0.17,data/raw/censo_escolar/microdados_ed_basica_20...
2,censo_escolar,2018,Escola.pdf,.pdf,0.18,data/raw/censo_escolar/microdados_ed_basica_20...
7,censo_escolar,2018,Leia-me.pdf,.pdf,0.17,data/raw/censo_escolar/microdados_ed_basica_20...
8,censo_escolar,2018,Nota.pdf,.pdf,0.11,data/raw/censo_escolar/microdados_ed_basica_20...
3,censo_escolar,2018,Profissional_Escolar.pdf,.pdf,0.18,data/raw/censo_escolar/microdados_ed_basica_20...
...,...,...,...,...,...,...
88,rendimento,2022,tx_rend_escolas_2022.ods,.ods,31.97,data/raw/rendimento/tx_rend_escolas_2022/tx_re...
89,rendimento,2022,tx_rend_escolas_2022.xlsx,.xlsx,38.19,data/raw/rendimento/tx_rend_escolas_2022/tx_re...
90,rendimento,2023,tx_rend_escolas_2023.ods,.ods,30.99,data/raw/rendimento/tx_rend_escolas_2023/tx_re...
91,rendimento,2023,tx_rend_escolas_2023.txt,.txt,0.00,data/raw/rendimento/tx_rend_escolas_2023/tx_re...


,base,ano,arquivos,tamanho_mb
0,censo_escolar,2018,9,215.05
1,censo_escolar,2019,10,207.66
2,censo_escolar,2020,10,204.00
3,censo_escolar,2021,10,201.44
4,censo_escolar,2022,9,181.84
5,censo_escolar,2023,10,204.82
6,media_alunos_turma,2018,2,43.06
7,media_alunos_turma,2019,3,54.38
8,media_alunos_turma,2020,3,42.13
9,media_alunos_turma,2021,3,41.95


## Localização e comparação dos arquivos
Este código cria funções para localizar automaticamente os arquivos de cada fonte e ano, priorizando os formatos adequados, como excel para os indicadores e CSV para os microdados.
Também permite inspecionar planilhas e comparar seus cabeçalhos com o layout de 2018, identificando colunas novas ou ausentes ao longo dos anos. Essa verificação ajuda a reconhecer mudanças estruturais antes do carregamento e da padronização dos dados.

In [3]:
def localizar_arquivo(base, ano, extensoes=None):
    extensoes = set(extensoes or [])
    candidatos = [
        p for p in (RAW / base).rglob(f"*{ano}*")
        if p.is_file() and (not extensoes or p.suffix.lower() in extensoes)
    ]
    if not candidatos:
        raise FileNotFoundError(f"Arquivo não encontrado: base={base}, ano={ano}")
    prioridade = {".xlsx": 0, ".csv": 0, ".ods": 1}
    return sorted(candidatos, key=lambda p: (prioridade.get(p.suffix.lower(), 9), len(str(p))))[0]

def inspecionar_excel(arquivo, planilha, header=8, nrows=5):
    amostra = pd.read_excel(arquivo, sheet_name=planilha, header=header, nrows=nrows, dtype=str)
    return {
        "arquivo": arquivo.name,
        "planilha": planilha,
        "n_colunas": len(amostra.columns),
        "colunas": list(amostra.columns),
        "amostra": amostra,
    }

def comparar_cabecalhos(base, planilha, anos=ANOS):
    registros = []
    conjuntos = {}
    for ano in anos:
        arquivo = localizar_arquivo(base, ano, {".xlsx"})
        info = inspecionar_excel(arquivo, planilha)
        conjuntos[ano] = set(info["colunas"])
        registros.append({"ano": ano, "arquivo": info["arquivo"], "n_colunas": info["n_colunas"]})
    referencia = conjuntos[anos[0]]
    comparacao = pd.DataFrame([
        {"ano": ano, "novas_vs_2018": sorted(cols - referencia), "ausentes_vs_2018": sorted(referencia - cols)}
        for ano, cols in conjuntos.items()
    ])
    return pd.DataFrame(registros), comparacao




##  Mapeamento e seleção das variáveis
Este código centraliza os nomes das colunas utilizadas em cada fonte e período, considerando as mudanças de layout ocorridas entre 2018 e 2023.
Os mapeamentos padronizam as variáveis de Rendimento e Média de Alunos, incluindo a particularidade dos campos invertidos em 2018. Também é definida a seleção das variáveis relevantes do Censo Escolar, abrangendo identificação, infraestrutura, tecnologia, acessibilidade, matrículas, docentes e turmas. Isso permite carregar apenas os dados necessários e manter uma estrutura consistente entre os anos

In [4]:
MAPA_RENDIMENTO = {
    ano: {
        "ano": "Ano" if ano <= 2020 else "NU_ANO_CENSO",
        "codigo_escola": "CO_ENTIDADE",
        "nome_escola": "NO_ENTIDADE",
        "municipio": "NO_MUNICIPIO",
        "uf": "SG_UF",
        "taxa_aprovacao": "tap_FUN" if ano <= 2020 else "1_CAT_FUN",
        "taxa_reprovacao": "tre_FUN" if ano <= 2020 else "2_CAT_FUN",
        "taxa_abandono": "tab_FUN" if ano <= 2020 else "3_CAT_FUN",
    } for ano in ANOS
}

MAPA_MEDIA = {
    2018: {
        "ano": "NU_ANO_CENSO", "codigo_escola": "NO_ENTIDADE",
        "nome_escola": "CO_ENTIDADE", "municipio": "CO_MUNICIPIO", "uf": "SG_UF",
        "media_alunos_fund": "ATU_FUN", "media_alunos_ai": "ATU_F14", "media_alunos_af": "ATU_F04",
    },
    **{
        ano: {
            "ano": "NU_ANO_CENSO", "codigo_escola": "CO_ENTIDADE",
            "nome_escola": "NO_ENTIDADE", "municipio": "NO_MUNICIPIO", "uf": "SG_UF",
            "media_alunos_fund": "FUN_CAT_0", "media_alunos_ai": "FUN_AI_CAT_0", "media_alunos_af": "FUN_AF_CAT_0",
        } for ano in ANOS if ano != 2018
    },
}

VARIAVEIS_CENSO = [
    "NU_ANO_CENSO", "CO_ENTIDADE", "NO_ENTIDADE", "CO_MUNICIPIO", "NO_MUNICIPIO", "SG_UF",
    "TP_DEPENDENCIA", "TP_LOCALIZACAO", "TP_SITUACAO_FUNCIONAMENTO", "IN_FUND",
    "IN_AGUA_POTAVEL", "IN_AGUA_REDE_PUBLICA", "IN_ENERGIA_REDE_PUBLICA", "IN_ESGOTO_REDE_PUBLICA", "IN_BANHEIRO",
    "IN_BIBLIOTECA", "IN_BIBLIOTECA_SALA_LEITURA", "IN_SALA_LEITURA", "IN_LABORATORIO_CIENCIAS", "IN_LABORATORIO_INFORMATICA",
    "IN_COMPUTADOR", "IN_DESKTOP_ALUNO", "IN_COMP_PORTATIL_ALUNO", "IN_TABLET_ALUNO", "IN_INTERNET", "IN_INTERNET_ALUNOS",
    "IN_INTERNET_APRENDIZAGEM", "IN_BANDA_LARGA", "IN_BANHEIRO_PNE", "IN_ACESSIBILIDADE_RAMPAS",
    "IN_ACESSIBILIDADE_CORRIMAO", "IN_ACESSIBILIDADE_PISOS_TATEIS", "IN_ACESSIBILIDADE_SINAL_SONORO",
    "IN_ACESSIBILIDADE_SINAL_TATIL", "IN_ACESSIBILIDADE_SINAL_VISUAL", "IN_ACESSIBILIDADE_INEXISTENTE",
    "QT_MAT_FUND", "QT_MAT_FUND_AI", "QT_MAT_FUND_AF", "QT_DOC_FUND", "QT_DOC_FUND_AI", "QT_DOC_FUND_AF",
    "QT_TUR_FUND", "QT_TUR_FUND_AI", "QT_TUR_FUND_AF",
]

## Validação das variáveis necessárias
Este código verifica se todas as colunas definidas nos mapeamentos estão presentes nos arquivos de Rendimento, Média de Alunos e Censo Escolar em cada ano analisado.
O resultado apresenta se cada arquivo possui a estrutura esperada e informa quais variáveis estão ausentes. Caso alguma coluna necessária não seja encontrada, a execução é interrompida para evitar que as próximas etapas sejam realizadas com dados incompletos ou layouts incompatíveis.

In [5]:
def validar_variaveis_excel(base, planilha, mapas):
    linhas = []
    for ano, mapa in mapas.items():
        arquivo = localizar_arquivo(base, ano, {".xlsx"})
        colunas = set(pd.read_excel(arquivo, sheet_name=planilha, header=8, nrows=0).columns)
        ausentes = sorted(set(mapa.values()) - colunas)
        linhas.append({"base": base, "ano": ano, "ok": not ausentes, "variaveis_ausentes": ausentes})
    return pd.DataFrame(linhas)

def validar_variaveis_censo(variaveis=VARIAVEIS_CENSO):
    linhas = []
    for ano in ANOS:
        arquivo = localizar_arquivo("censo_escolar", ano, {".csv"})
        colunas = set(pd.read_csv(arquivo, sep=";", encoding="latin1", nrows=0).columns)
        ausentes = sorted(set(variaveis) - colunas)
        linhas.append({"base": "censo_escolar", "ano": ano, "ok": not ausentes, "variaveis_ausentes": ausentes})
    return pd.DataFrame(linhas)

validacao_variaveis = pd.concat([
    validar_variaveis_excel("rendimento", "ESCOLAS", MAPA_RENDIMENTO),
    validar_variaveis_excel("media_alunos_turma", "ESCOLA", MAPA_MEDIA),
    validar_variaveis_censo(),
], ignore_index=True)
display(validacao_variaveis)
assert validacao_variaveis["ok"].all(), "Existem variáveis esperadas ausentes; revise os layouts antes de prosseguir."

,base,ano,ok,variaveis_ausentes
0,rendimento,2018,True,[]
1,rendimento,2019,True,[]
2,rendimento,2020,True,[]
3,rendimento,2021,True,[]
4,rendimento,2022,True,[]
5,rendimento,2023,True,[]
6,media_alunos_turma,2018,True,[]
7,media_alunos_turma,2019,True,[]
8,media_alunos_turma,2020,True,[]
9,media_alunos_turma,2021,True,[]


## Funções de carregamento e padronização
Este código reúne funções reutilizáveis para carregar as bases de Rendimento, Média de Alunos e Censo Escolar entre 2018 e 2023.
Durante o carregamento, as colunas são renomeadas conforme os mapeamentos definidos, os códigos das escolas são normalizados e os registros são filtrados para Porto Alegre. O Censo é lido em blocos para reduzir o uso de memória. Também são adicionados o ano e o arquivo de origem, garantindo rastreabilidade e uma estrutura padronizada para as etapas seguintes.

In [6]:
def normalizar_codigo(serie):
    return serie.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def renomear_por_mapa(df, mapa):
    return df.rename(columns={origem: destino for destino, origem in mapa.items()})

def filtrar_porto_alegre(df, coluna_uf="uf", coluna_municipio="municipio"):
    return df[
        df[coluna_uf].astype("string").str.strip().eq(UF_ALVO)
        & df[coluna_municipio].astype("string").str.strip().eq(MUNICIPIO_ALVO)
    ].copy()

def carregar_rendimento(ano):
    mapa = MAPA_RENDIMENTO[ano]
    arquivo = localizar_arquivo("rendimento", ano, {".xlsx"})
    df = pd.read_excel(arquivo, sheet_name="ESCOLAS", header=8, dtype=str, usecols=list(mapa.values()))
    df = filtrar_porto_alegre(renomear_por_mapa(df, mapa))
    df["ano"] = int(ano)
    df["codigo_escola"] = normalizar_codigo(df["codigo_escola"])
    df["arquivo_origem"] = str(arquivo.relative_to(PROJECT_ROOT))
    return df

def carregar_media_alunos(ano):
    mapa = MAPA_MEDIA[ano]
    arquivo = localizar_arquivo("media_alunos_turma", ano, {".xlsx"})
    df = pd.read_excel(arquivo, sheet_name="ESCOLA", header=8, dtype=str, usecols=list(mapa.values()))
    df = filtrar_porto_alegre(renomear_por_mapa(df, mapa))
    df["ano"] = int(ano)
    df["codigo_escola"] = normalizar_codigo(df["codigo_escola"])
    df["arquivo_origem"] = str(arquivo.relative_to(PROJECT_ROOT))
    return df

def carregar_censo(ano, variaveis=VARIAVEIS_CENSO):
    arquivo = localizar_arquivo("censo_escolar", ano, {".csv"})
    partes = []
    for bloco in pd.read_csv(
        arquivo, sep=";", encoding="latin1", usecols=variaveis, dtype=str,
        chunksize=CHUNK_SIZE, low_memory=False
    ):
        parte = bloco[
            bloco["SG_UF"].astype("string").str.strip().eq(UF_ALVO)
            & bloco["NO_MUNICIPIO"].astype("string").str.strip().eq(MUNICIPIO_ALVO)
        ]
        if not parte.empty:
            partes.append(parte.copy())
    df = pd.concat(partes, ignore_index=True)
    df = df.rename(columns={
        "NU_ANO_CENSO": "ano", "CO_ENTIDADE": "codigo_escola",
        "NO_ENTIDADE": "nome_escola", "NO_MUNICIPIO": "municipio", "SG_UF": "uf",
    })
    df["ano"] = int(ano)
    df["codigo_escola"] = normalizar_codigo(df["codigo_escola"])
    df["arquivo_origem"] = str(arquivo.relative_to(PROJECT_ROOT))
    return df

def carregar_periodo(carregador, anos=ANOS):
    return pd.concat([carregador(ano) for ano in anos], ignore_index=True)

## Carregamento consolidado das bases
Este código carrega e reúne os dados de 2018 a 2023 das três fontes utilizadas no projeto: Rendimento, Média de Alunos por Turma e Censo Escolar.
Cada fonte é lida apenas uma vez e armazenada em um DataFrame para ser reutilizada nas validações seguintes. Ao final, é exibido um resumo com a quantidade de registros de Porto Alegre e o número de colunas de cada base.

In [7]:
rendimento = carregar_periodo(carregar_rendimento)
media_alunos = carregar_periodo(carregar_media_alunos)
censo = carregar_periodo(carregar_censo)

display(pd.DataFrame({
    "base": ["rendimento", "media_alunos_turma", "censo_escolar"],
    "registros_porto_alegre": [len(rendimento), len(media_alunos), len(censo)],
    "colunas": [rendimento.shape[1], media_alunos.shape[1], censo.shape[1]],
}))

,base,registros_porto_alegre,colunas
0,rendimento,2292,9
1,media_alunos_turma,5651,9
2,censo_escolar,6475,46


## Auditoria de qualidade dos indicadores
Este código verifica a qualidade das variáveis de Rendimento e Média de Alunos por Turma em cada ano.
A auditoria identifica valores nulos, marcadores --, registros duplicados na chave ano + codigo_escola e valores fora das faixas esperadas. Ao final, são exibidos um resumo geral da qualidade e uma tabela detalhada com as variáveis que apresentam dados ausentes.

In [8]:
COLUNAS_RENDIMENTO = ["taxa_aprovacao", "taxa_reprovacao", "taxa_abandono"]
COLUNAS_MEDIA = ["media_alunos_fund", "media_alunos_ai", "media_alunos_af"]

def contar_duplicados_chave(df):
    return int(df.duplicated(["ano", "codigo_escola"], keep=False).sum())

def detalhar_ausencias(df, colunas, base):
    linhas = []
    for ano, grupo in df.groupby("ano"):
        for coluna in colunas:
            texto = grupo[coluna].astype("string").str.strip()
            nulos = int(grupo[coluna].isna().sum())
            tracos = int(texto.eq("--").sum())
            linhas.append({
                "base": base, "ano": ano, "variavel": coluna,
                "nulos": nulos, "valores_traco": tracos,
                "ausencias_total": nulos + tracos,
                "percentual": round((nulos + tracos) / len(grupo) * 100, 2) if len(grupo) else 0,
            })
    return pd.DataFrame(linhas)

def contar_fora_faixa(df, colunas, minimo=0, maximo=None):
    valores = df[colunas].replace("--", pd.NA).apply(pd.to_numeric, errors="coerce")
    invalidos = valores.lt(minimo)
    if maximo is not None:
        invalidos |= valores.gt(maximo)
    return int(invalidos.sum().sum())

def resumo_auditoria_indicador(df, base, colunas, maximo=None):
    linhas = []
    for ano, grupo in df.groupby("ano"):
        texto = grupo[colunas].astype("string").apply(lambda s: s.str.strip())
        linhas.append({
            "base": base, "ano": int(ano), "registros_porto_alegre": len(grupo),
            "nulos": int(grupo[colunas].isna().sum().sum()),
            "valores_traco": int(texto.eq("--").sum().sum()),
            "duplicados_ano_escola": contar_duplicados_chave(grupo),
            "valores_fora_faixa": contar_fora_faixa(grupo, colunas, 0, maximo),
        })
    return pd.DataFrame(linhas)

resumo_indicadores = pd.concat([
    resumo_auditoria_indicador(rendimento, "rendimento", COLUNAS_RENDIMENTO, 100),
    resumo_auditoria_indicador(media_alunos, "media_alunos_turma", COLUNAS_MEDIA),
], ignore_index=True)

detalhes_ausencias = pd.concat([
    detalhar_ausencias(rendimento, COLUNAS_RENDIMENTO, "rendimento"),
    detalhar_ausencias(media_alunos, COLUNAS_MEDIA, "media_alunos_turma"),
], ignore_index=True)

display(resumo_indicadores)
display(detalhes_ausencias.query("ausencias_total > 0").sort_values(
    ["base", "ano", "ausencias_total"], ascending=[True, True, False]
))

,base,ano,registros_porto_alegre,nulos,valores_traco,duplicados_ano_escola,valores_fora_faixa
0,rendimento,2018,385,0,84,0,0
1,rendimento,2019,382,0,87,0,0
2,rendimento,2020,381,0,99,0,0
3,rendimento,2021,380,0,126,0,0
4,rendimento,2022,381,0,120,0,0
5,rendimento,2023,383,0,111,0,0
6,media_alunos_turma,2018,987,0,1924,0,0
7,media_alunos_turma,2019,973,0,1888,0,0
8,media_alunos_turma,2020,950,0,1822,0,0
9,media_alunos_turma,2021,923,0,1749,0,0


,base,ano,variavel,nulos,valores_traco,ausencias_total,percentual
20,media_alunos_turma,2018,media_alunos_af,0,659,659,66.77
19,media_alunos_turma,2018,media_alunos_ai,0,637,637,64.54
18,media_alunos_turma,2018,media_alunos_fund,0,628,628,63.63
23,media_alunos_turma,2019,media_alunos_af,0,645,645,66.29
22,media_alunos_turma,2019,media_alunos_ai,0,625,625,64.23
21,media_alunos_turma,2019,media_alunos_fund,0,618,618,63.51
26,media_alunos_turma,2020,media_alunos_af,0,623,623,65.58
25,media_alunos_turma,2020,media_alunos_ai,0,603,603,63.47
24,media_alunos_turma,2020,media_alunos_fund,0,596,596,62.74
29,media_alunos_turma,2021,media_alunos_af,0,596,596,64.57


## Auditoria de qualidade do Censo Escolar
Este código realiza uma verificação anual da qualidade dos dados do Censo Escolar de Porto Alegre.
A auditoria contabiliza valores nulos, registros duplicados na chave ano + codigo_escola e valores inválidos nas variáveis iniciadas por IN_, que devem ser indicadores binários representados por 0 ou 1. O resultado permite identificar possíveis problemas antes da aplicação do recorte analítico e das etapas de integração.

In [9]:
def auditar_censo(df):
    linhas = []
    colunas_in = [c for c in df.columns if c.startswith("IN_")]
    for ano, grupo in df.groupby("ano"):
        numericos = grupo[colunas_in].apply(pd.to_numeric, errors="coerce")
        invalidos = numericos.notna() & ~numericos.isin([0, 1])
        linhas.append({
            "base": "censo_escolar", "ano": int(ano),
            "registros_porto_alegre": len(grupo),
            "nulos": int(grupo.isna().sum().sum()),
            "duplicados_ano_escola": contar_duplicados_chave(grupo),
            "indicadores_binarios_invalidos": int(invalidos.sum().sum()),
        })
    return pd.DataFrame(linhas)

resumo_censo = auditar_censo(censo)
display(resumo_censo)

,base,ano,registros_porto_alegre,nulos,duplicados_ano_escola,indicadores_binarios_invalidos
0,censo_escolar,2018,1150,0,0,0
1,censo_escolar,2019,1133,0,0,0
2,censo_escolar,2020,1054,0,0,0
3,censo_escolar,2021,1047,0,0,0
4,censo_escolar,2022,1102,5192,0,0
5,censo_escolar,2023,989,1447,0,0


## Aplicação e validação do recorte do projeto
Este código seleciona, entre as escolas de Porto Alegre, somente as instituições públicas que oferecem Ensino Fundamental. São consideradas as dependências Federal, Estadual e Municipal.
Em seguida, apresenta a quantidade de escolas por ano e dependência administrativa, além do total anual do universo analisado. Por fim, verifica se cada combinação de ano + codigo_escola é única, interrompendo a execução caso existam registros duplicados.

In [10]:
def aplicar_recorte_projeto(df_censo):
    dependencia = df_censo["TP_DEPENDENCIA"].astype("string").str.strip()
    fundamental = df_censo["IN_FUND"].astype("string").str.strip()
    return df_censo[dependencia.isin(DEPENDENCIAS_PUBLICAS) & fundamental.eq("1")].copy()

censo_recorte = aplicar_recorte_projeto(censo)

resumo_recorte = (
    censo_recorte.groupby(["ano", "TP_DEPENDENCIA"], dropna=False)
    .size().rename("escolas").reset_index()
    .pivot(index="ano", columns="TP_DEPENDENCIA", values="escolas").fillna(0).astype(int)
    .rename(columns={"1": "federal", "2": "estadual", "3": "municipal"})
)
resumo_recorte["total"] = resumo_recorte.sum(axis=1)
display(resumo_recorte.reset_index())

assert contar_duplicados_chave(censo_recorte) == 0, "O recorte possui chave ano + escola duplicada."

TP_DEPENDENCIA,ano,federal,estadual,municipal,total
0,2018,2,220,54,276
1,2019,2,216,54,272
2,2020,2,215,54,271
3,2021,2,212,54,268
4,2022,2,211,54,267
5,2023,2,211,54,267


## Validação da cobertura entre as bases
Este código compara o universo de escolas definido pelo Censo Escolar com os registros disponíveis nas bases de Rendimento e Média de Alunos por Turma.
Para cada ano, são contabilizadas as escolas presentes em cada fonte, aquelas com indicadores disponíveis, as ausentes da base e as que possuem registro, mas não têm informação válida. As pendências são mantidas em uma tabela detalhada, permitindo documentar diferenças de cobertura sem preencher ou excluir dados silenciosamente.

In [11]:
def indicador_disponivel(df, colunas):
    texto = df[colunas].astype("string").apply(lambda s: s.str.strip())
    return texto.notna().any(axis=1) & ~texto.eq("--").all(axis=1)

def comparar_cobertura(censo_recorte, indicador, colunas_indicador, nome_base):
    linhas = []
    detalhes = []
    for ano in ANOS:
        universo = censo_recorte.loc[censo_recorte["ano"].eq(ano), ["codigo_escola", "nome_escola"]].drop_duplicates()
        base_ano = indicador[indicador["ano"].eq(ano)].copy()
        base_ano["indicador_disponivel"] = indicador_disponivel(base_ano, colunas_indicador)
        comparacao = universo.merge(
            base_ano[["codigo_escola", "indicador_disponivel"]].drop_duplicates("codigo_escola"),
            on="codigo_escola", how="left", indicator=True
        )
        comparacao["presente_na_base"] = comparacao["_merge"].eq("both")
        comparacao["indicador_disponivel"] = comparacao["indicador_disponivel"].fillna(False).astype(bool)
        linhas.append({
            "base": nome_base, "ano": ano, "escolas_no_recorte": len(universo),
            "presentes_na_base": int(comparacao["presente_na_base"].sum()),
            "com_indicador": int(comparacao["indicador_disponivel"].sum()),
            "ausentes_da_base": int((~comparacao["presente_na_base"]).sum()),
            "presentes_sem_indicador": int((comparacao["presente_na_base"] & ~comparacao["indicador_disponivel"]).sum()),
        })
        pendencias = comparacao[~comparacao["indicador_disponivel"]].copy()
        pendencias.insert(0, "base", nome_base)
        pendencias.insert(1, "ano", ano)
        detalhes.append(pendencias.drop(columns="_merge"))
    return pd.DataFrame(linhas), pd.concat(detalhes, ignore_index=True)

cobertura_rendimento, pendencias_rendimento = comparar_cobertura(
    censo_recorte, rendimento, COLUNAS_RENDIMENTO, "rendimento"
)
cobertura_media, pendencias_media = comparar_cobertura(
    censo_recorte, media_alunos, COLUNAS_MEDIA, "media_alunos_turma"
)
cobertura = pd.concat([cobertura_rendimento, cobertura_media], ignore_index=True)
pendencias_cobertura = pd.concat([pendencias_rendimento, pendencias_media], ignore_index=True)

display(cobertura)

,base,ano,escolas_no_recorte,presentes_na_base,com_indicador,ausentes_da_base,presentes_sem_indicador
0,rendimento,2018,276,268,268,8,0
1,rendimento,2019,272,264,264,8,0
2,rendimento,2020,271,263,263,8,0
3,rendimento,2021,268,260,258,8,2
4,rendimento,2022,267,259,259,8,0
5,rendimento,2023,267,259,259,8,0
6,media_alunos_turma,2018,276,268,268,8,0
7,media_alunos_turma,2019,272,264,264,8,0
8,media_alunos_turma,2020,271,263,263,8,0
9,media_alunos_turma,2021,268,260,260,8,0


## Identificação de ausências sistemáticas e exceções
Este código identifica escolas que permaneceram sem indicadores de Rendimento ou Média de Alunos em todos os anos analisados, permitindo diferenciar ausências recorrentes de problemas pontuais.
Também examina especificamente as escolas federais de 2021 para verificar a indisponibilidade das taxas de rendimento já identificada anteriormente. Essa análise documenta exceções sem substituir valores ausentes por zero ou realizar imputações.

In [12]:
total_anos = len(ANOS)
sistematicamente_ausentes = (
    pendencias_cobertura.groupby(["base", "codigo_escola", "nome_escola"])
    .agg(anos_sem_indicador=("ano", "nunique"), anos=("ano", lambda s: sorted(set(s))))
    .reset_index()
    .query("anos_sem_indicador == @total_anos")
)
display(sistematicamente_ausentes)

federais_2021 = censo_recorte[
    censo_recorte["ano"].eq(2021)
    & censo_recorte["TP_DEPENDENCIA"].astype("string").str.strip().eq("1")
][["codigo_escola", "nome_escola"]]
excecao_federal_2021 = federais_2021.merge(
    rendimento[rendimento["ano"].eq(2021)][["codigo_escola", *COLUNAS_RENDIMENTO]],
    on="codigo_escola", how="left"
)
display(excecao_federal_2021)

,base,codigo_escola,nome_escola,anos_sem_indicador,anos
0,media_alunos_turma,43174329,ESC EST ESPEC RECANTO DA ALEGRIA,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
1,media_alunos_turma,43174337,ESC EST ESPEC CRISTO REDENTOR,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
2,media_alunos_turma,43174345,ESC EST ESPEC RENASCENCA,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
3,media_alunos_turma,43178995,EMEEF PROF LYGIA MORRONE AVERBUCK,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
4,media_alunos_turma,43179002,EMEEF PROF ELYSEU PAGLIOLI,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
5,media_alunos_turma,43179010,EMEEF PROF LUIZ FRANCISCO LUCENA BORGES,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
6,media_alunos_turma,43179029,EMEEF TRISTAO SUCUPIRA VIANNA,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
7,media_alunos_turma,43180523,ESC EST ENS MED PARA SURDOS PROF LILIA MAZERON,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
10,rendimento,43174329,ESC EST ESPEC RECANTO DA ALEGRIA,6,"[2018, 2019, 2020, 2021, 2022, 2023]"
11,rendimento,43174337,ESC EST ESPEC CRISTO REDENTOR,6,"[2018, 2019, 2020, 2021, 2022, 2023]"


,codigo_escola,nome_escola,taxa_aprovacao,taxa_reprovacao,taxa_abandono
0,43104932,COLEGIO DE APLICACAO UFRGS,--,--,--
1,43105009,COLEGIO MILITAR DE PORTO ALEGRE,--,--,--


## Validação da consistência das taxas de rendimento
Este código verifica se a soma das taxas de aprovação, reprovação e abandono corresponde a 100% em cada registro que possui os três indicadores disponíveis.
A validação apresenta, por ano, a quantidade de registros completos, o número de somas inconsistentes e os valores mínimo e máximo encontrados. Caso alguma inconsistência seja identificada, a execução é interrompida para que os registros sejam investigados antes das próximas etapas.

In [13]:
def validar_consistencia_rendimento(df, tolerancia=0.01):
    taxas = df[COLUNAS_RENDIMENTO].replace("--", pd.NA).apply(pd.to_numeric, errors="coerce")
    completos = taxas.notna().all(axis=1)
    soma = taxas.sum(axis=1, min_count=3)
    inconsistente = completos & ~soma.sub(100).abs().le(tolerancia)
    resumo = []
    for ano in ANOS:
        mascara = df["ano"].eq(ano)
        somas_ano = soma[mascara & completos]
        resumo.append({
            "ano": ano,
            "registros_porto_alegre": int(mascara.sum()),
            "registros_com_3_taxas": int((mascara & completos).sum()),
            "soma_diferente_100": int((mascara & inconsistente).sum()),
            "soma_minima": somas_ano.min(),
            "soma_maxima": somas_ano.max(),
        })
    detalhes = df.loc[inconsistente, ["ano", "codigo_escola", "nome_escola", *COLUNAS_RENDIMENTO]].copy()
    detalhes["soma_taxas"] = soma[inconsistente]
    return pd.DataFrame(resumo), detalhes

consistencia_rendimento, inconsistencias_rendimento = validar_consistencia_rendimento(rendimento)
display(consistencia_rendimento)
assert inconsistencias_rendimento.empty, "Foram encontradas somas de rendimento diferentes de 100%."

,ano,registros_porto_alegre,registros_com_3_taxas,soma_diferente_100,soma_minima,soma_maxima
0,2018,385,357,0,100.0,100.0
1,2019,382,353,0,100.0,100.0
2,2020,381,348,0,100.0,100.0
3,2021,380,338,0,100.0,100.0
4,2022,381,341,0,100.0,100.0
5,2023,383,346,0,100.0,100.0


## Conclusões da compreensão e da auditoria
A compreensão e a auditoria confirmaram as principais regras para a preparação e futura integração das bases:
- A chave analítica do projeto é ano + codigo_escola.
- O recorte considera escolas de Porto Alegre das redes Federal, Estadual e Municipal que oferecem Ensino Fundamental.
- Valores nulos e o marcador -- representam ausência de informação, nunca valor zero.
- A cobertura das fontes deve ser comparada ao universo anual do Censo Escolar, pois a composição das escolas varia ao longo do período.
- Os anos de 2020 e 2021 devem ser contextualizados devido à pandemia, sem pressupor causalidade ou erro nos dados.
- Quando disponíveis, as taxas de aprovação, reprovação e abandono devem totalizar 100%.
- As ausências identificadas serão preservadas e documentadas, sem imputação ou exclusão silenciosa.